### 1. Projection du Taux de Pauvreté en 2023 à l'Aide d'un Modèle de Régression Logistique Entraîné sur les Données de 2018

In [ ]:
import pandas as pd
import os
import pickle

# Définition des chemins
input_csv_2023 = r"D:\wealth_predict_sentinel\Data\processed_csv\images_with_features_extracted_4096_fullyConnected_sans_augm_couche_nongelee_batch_16_base_EHCVM_2018_pour_an_2023.csv"
model_path = r"D:\wealth_predict_sentinel\models\fullyConnected_sans_augm_couche_nongelee_batch_16_logistic_pca_5-fold_logistic_reg.pkl"
scaler_path = r"D:\wealth_predict_sentinel\models\scaler_fullyConnected_sans_augm_couche_nongelee_batch_16_cross_validation.pkl"
pca_path = r"D:\wealth_predict_sentinel\models\pca_fullyConnected_sans_augm_couche_nongelee_batch_16_cross_validation.pkl"
output_dir_csv = r"D:\wealth_predict_sentinel\Data\processed_csv"

# Chargement des données de 2023
df_2023 = pd.read_csv(input_csv_2023)

# Vérification des colonnes nécessaires
features = [f'feature_{i}' for i in range(4096)]
required_columns = features + ['hhweight', 'hhsize']  # Colonnes nécessaires pour le calcul du taux de pauvreté
missing_columns = [col for col in required_columns if col not in df_2023.columns]

if missing_columns:
    raise ValueError(f"Les colonnes suivantes sont manquantes dans les données de 2023 : {missing_columns}")

X_2023 = df_2023[features]

# Chargement du scaler et du PCA utilisés en 2018
with open(scaler_path, 'rb') as file:
    scaler = pickle.load(file)

with open(pca_path, 'rb') as file:
    pca = pickle.load(file)

# Normalisation des données 2023 avec le même scaler
X_2023_scaled = scaler.transform(X_2023)

# Réduction de dimension avec le même PCA
X_2023_pca = pca.transform(X_2023_scaled)

# Chargement du modèle entraîné en 2018
# La ligne suivante charge le modèle avec ses hyperparamètres (class_weight='balanced')
with open(model_path, 'rb') as file:
    logistic_model = pickle.load(file)

# Prédictions sur les données de 2023
y_pred_2023 = logistic_model.predict(X_2023_pca)

# Ajout des prédictions au DataFrame
df_2023['pcexp_binaire_pred'] = y_pred_2023

# Calcul du taux de pauvreté prévisionnel pour 2023 en utilisant les pondérations de 2018
weighted_predicted_poverty_2023 = (
    (df_2023['hhweight'] * df_2023['hhsize'] * df_2023['pcexp_binaire_pred']).sum() /
    (df_2023['hhweight'] * df_2023['hhsize']).sum()
)

print(f"Taux de pauvreté prévisionnel en 2023 : {weighted_predicted_poverty_2023:.4%}")

# Sauvegarde des résultats finaux
final_csv_path = os.path.join(output_dir_csv, "fullyConnected_sans_augm_couche_nongelee_batch_16_test_with_poverty_2023_logistic_reg.csv")
df_2023.to_csv(final_csv_path, index=False)

print(f"Les prédictions ont été sauvegardées dans : {final_csv_path}")

In [ ]:
import pandas as pd
import numpy as np
import os
import time
import psutil
import gc
from joblib import Parallel, delayed
from scipy.stats import norm

# Démarrer le chronomètre
start_time = time.time()

# Définition des chemins
input_csv_2023 = r"D:\wealth_predict_sentinel\Data\processed_csv\fullyConnected_sans_augm_couche_nongelee_batch_16_test_with_poverty_2023_logistic_reg.csv"
output_dir_csv = r"D:\wealth_predict_sentinel\Data\processed_csv"

# Chargement des résultats de 2023
df_2023 = pd.read_csv(input_csv_2023)

# Vérification des colonnes nécessaires
required_columns = ['hhweight', 'hhsize', 'pcexp_binaire_pred']

missing_columns = [col for col in required_columns if col not in df_2023.columns]
if missing_columns:
    raise ValueError(f"Les colonnes suivantes sont manquantes dans les données de 2023 : {missing_columns}")

# Nombre total d'itérations bootstrap
n_bootstrap = 1000
batch_size = 100  # Taille du batch pour optimiser la mémoire
n_cores = 12  # Nombre de cœurs à utiliser pour le traitement parallèle

# Fonction pour exécuter une itération bootstrap
def bootstrap_iteration(seed, df):
    """Effectue une itération bootstrap et retourne le taux de pauvreté"""
    np.random.seed(seed)
    sample_indices = np.random.choice(df.index, size=len(df), replace=True)
    sample_df = df.loc[sample_indices]

    # Calcul du taux de pauvreté pondéré pour cet échantillon bootstrap
    weighted_poverty = (
        (sample_df['hhweight'] * sample_df['hhsize'] * sample_df['pcexp_binaire_pred']).sum() /
        (sample_df['hhweight'] * sample_df['hhsize']).sum()
    )
    return weighted_poverty

# Liste pour stocker les résultats bootstrap
bootstrap_poverty_rates = []

# Découpage en batches pour optimiser la mémoire
n_batches = (n_bootstrap + batch_size - 1) // batch_size

for batch in range(n_batches):
    start_idx = batch * batch_size
    end_idx = min((batch + 1) * batch_size, n_bootstrap)

    # Vérification de la mémoire disponible
    available_memory = psutil.virtual_memory().available / (1024 * 1024 * 1024)  # en Go
    print(f"Mémoire disponible: {available_memory:.2f} GB")

    # Pause si la mémoire devient insuffisante
    if available_memory < 2:
        print("⚠ Mémoire faible, pause de 10 secondes pour libérer de l'espace...")
        time.sleep(10)
        gc.collect()

    # Exécution parallèle du bootstrap
    seeds = range(start_idx, end_idx)
    batch_results = Parallel(n_jobs=n_cores)(
        delayed(bootstrap_iteration)(seed, df_2023) for seed in seeds
    )

    # Ajout des résultats
    bootstrap_poverty_rates.extend(batch_results)

    # Libération de la mémoire
    gc.collect()
    print(f"Batch {batch + 1}/{n_batches} terminé. Résultats accumulés: {len(bootstrap_poverty_rates)}")

# Conversion en tableau NumPy
bootstrap_poverty_rates = np.array(bootstrap_poverty_rates)

# Calcul des statistiques
mean_poverty = np.mean(bootstrap_poverty_rates)
std_poverty = np.std(bootstrap_poverty_rates, ddof=1)

# Intervalle de confiance à 95%
ci_lower = np.percentile(bootstrap_poverty_rates, 2.5)
ci_upper = np.percentile(bootstrap_poverty_rates, 97.5)

# Affichage des résultats
print(f"\n📊 Résumé des résultats:")
print(f"Taux de pauvreté prévisionnel moyen en 2023 : {mean_poverty:.4%}")
print(f"Intervalle de confiance à 95% : [{ci_lower:.4%}, {ci_upper:.4%}]")
print(f"Erreur standard : {std_poverty:.4%}")

# Sauvegarde des résultats dans un fichier CSV
bootstrap_results_path = os.path.join(output_dir_csv, "bootstrap_poverty_2023_results.csv")
pd.DataFrame({"bootstrap_poverty_rate": bootstrap_poverty_rates}).to_csv(bootstrap_results_path, index=False)

print(f"\n✅ Les résultats bootstrap ont été sauvegardés dans : {bootstrap_results_path}")

# Mesurer le temps d'exécution total
end_time = time.time()
elapsed_time = end_time - start_time

# Affichage du temps d'exécution
print("\n🔹 Temps total d'exécution :")
print(f"⏳ {elapsed_time:.2f} secondes")
print(f"⏳ {elapsed_time / 60:.2f} minutes")

### Essai avec le modèle basée sur le regular split 50%-50%

In [ ]:
import pandas as pd
import os
import pickle

# Définition des chemins
input_csv_2023 = r"D:\wealth_predict_sentinel\Data\processed_csv\images_with_features_extracted_4096_fullyConnected_sans_augm_couche_nongelee_batch_16_base_EHCVM_2018_pour_an_2023.csv"
model_path = r"D:\wealth_predict_sentinel\models\fullyConnected_sans_augm_couche_nongelee_batch_16_logistic_pca.pkl"
scaler_path = r"D:\wealth_predict_sentinel\models\scaler_fullyConnected_sans_augm_couche_nongelee_batch_16_regular_split.pkl"
pca_path = r"D:\wealth_predict_sentinel\models\pca_fullyConnected_sans_augm_couche_nongelee_batch_16_regular_split.pkl"
output_dir_csv = r"D:\wealth_predict_sentinel\Data\processed_csv"

# Chargement des données de 2023
df_2023 = pd.read_csv(input_csv_2023)

# Vérification des colonnes nécessaires
features = [f'feature_{i}' for i in range(4096)]
required_columns = features + ['hhweight', 'hhsize']  # Colonnes nécessaires pour le calcul du taux de pauvreté
missing_columns = [col for col in required_columns if col not in df_2023.columns]

if missing_columns:
    raise ValueError(f"Les colonnes suivantes sont manquantes dans les données de 2023 : {missing_columns}")

X_2023 = df_2023[features]

# Chargement du scaler et du PCA utilisés en 2018
with open(scaler_path, 'rb') as file:
    scaler = pickle.load(file)

with open(pca_path, 'rb') as file:
    pca = pickle.load(file)

# Normalisation des données 2023 avec le même scaler
X_2023_scaled = scaler.transform(X_2023)

# Réduction de dimension avec le même PCA
X_2023_pca = pca.transform(X_2023_scaled)

# Chargement du modèle entraîné en 2018
# La ligne suivante charge le modèle avec ses hyperparamètres (class_weight='balanced')
with open(model_path, 'rb') as file:
    logistic_model = pickle.load(file)

# Prédictions sur les données de 2023
y_pred_2023 = logistic_model.predict(X_2023_pca)

# Ajout des prédictions au DataFrame
df_2023['pcexp_binaire_pred'] = y_pred_2023

# Calcul du taux de pauvreté prévisionnel pour 2023 en utilisant les pondérations de 2018
weighted_predicted_poverty_2023 = (
    (df_2023['hhweight'] * df_2023['hhsize'] * df_2023['pcexp_binaire_pred']).sum() /
    (df_2023['hhweight'] * df_2023['hhsize']).sum()
)

print(f"Taux de pauvreté prévisionnel en 2023 : {weighted_predicted_poverty_2023:.4%}")

# Sauvegarde des résultats finaux
final_csv_path = os.path.join(output_dir_csv, "fullyConnected_sans_augm_couche_nongelee_batch_16_test_with_poverty_2023_logistic_reg_reg_regular_split.csv")
df_2023.to_csv(final_csv_path, index=False)

print(f"Les prédictions ont été sauvegardées dans : {final_csv_path}")

In [ ]:
import pandas as pd
import numpy as np
import os
import time
import psutil
import gc
from joblib import Parallel, delayed
from scipy.stats import norm

# Démarrer le chronomètre
start_time = time.time()

# Définition des chemins
input_csv_2023 = r"D:\wealth_predict_sentinel\Data\processed_csv\fullyConnected_sans_augm_couche_nongelee_batch_16_test_with_poverty_2023_logistic_reg_reg_regular_split.csv"
output_dir_csv = r"D:\wealth_predict_sentinel\Data\processed_csv"

# Chargement des résultats de 2023
df_2023 = pd.read_csv(input_csv_2023)

# Vérification des colonnes nécessaires
required_columns = ['hhweight', 'hhsize', 'pcexp_binaire_pred']

missing_columns = [col for col in required_columns if col not in df_2023.columns]
if missing_columns:
    raise ValueError(f"Les colonnes suivantes sont manquantes dans les données de 2023 : {missing_columns}")

# Nombre total d'itérations bootstrap
n_bootstrap = 1000
batch_size = 100  # Taille du batch pour optimiser la mémoire
n_cores = 12  # Nombre de cœurs à utiliser pour le traitement parallèle

# Fonction pour exécuter une itération bootstrap
def bootstrap_iteration(seed, df):
    """Effectue une itération bootstrap et retourne le taux de pauvreté"""
    np.random.seed(seed)
    sample_indices = np.random.choice(df.index, size=len(df), replace=True)
    sample_df = df.loc[sample_indices]

    # Calcul du taux de pauvreté pondéré pour cet échantillon bootstrap
    weighted_poverty = (
        (sample_df['hhweight'] * sample_df['hhsize'] * sample_df['pcexp_binaire_pred']).sum() /
        (sample_df['hhweight'] * sample_df['hhsize']).sum()
    )
    return weighted_poverty

# Liste pour stocker les résultats bootstrap
bootstrap_poverty_rates = []

# Découpage en batches pour optimiser la mémoire
n_batches = (n_bootstrap + batch_size - 1) // batch_size

for batch in range(n_batches):
    start_idx = batch * batch_size
    end_idx = min((batch + 1) * batch_size, n_bootstrap)

    # Vérification de la mémoire disponible
    available_memory = psutil.virtual_memory().available / (1024 * 1024 * 1024)  # en Go
    print(f"Mémoire disponible: {available_memory:.2f} GB")

    # Pause si la mémoire devient insuffisante
    if available_memory < 2:
        print("⚠ Mémoire faible, pause de 10 secondes pour libérer de l'espace...")
        time.sleep(10)
        gc.collect()

    # Exécution parallèle du bootstrap
    seeds = range(start_idx, end_idx)
    batch_results = Parallel(n_jobs=n_cores)(
        delayed(bootstrap_iteration)(seed, df_2023) for seed in seeds
    )

    # Ajout des résultats
    bootstrap_poverty_rates.extend(batch_results)

    # Libération de la mémoire
    gc.collect()
    print(f"Batch {batch + 1}/{n_batches} terminé. Résultats accumulés: {len(bootstrap_poverty_rates)}")

# Conversion en tableau NumPy
bootstrap_poverty_rates = np.array(bootstrap_poverty_rates)

# Calcul des statistiques
mean_poverty = np.mean(bootstrap_poverty_rates)
std_poverty = np.std(bootstrap_poverty_rates, ddof=1)

# Intervalle de confiance à 95%
ci_lower = np.percentile(bootstrap_poverty_rates, 2.5)
ci_upper = np.percentile(bootstrap_poverty_rates, 97.5)

# Affichage des résultats
print(f"\n📊 Résumé des résultats:")
print(f"Taux de pauvreté prévisionnel moyen en 2023 : {mean_poverty:.4%}")
print(f"Intervalle de confiance à 95% : [{ci_lower:.4%}, {ci_upper:.4%}]")
print(f"Erreur standard : {std_poverty:.4%}")

# Sauvegarde des résultats dans un fichier CSV
bootstrap_results_path = os.path.join(output_dir_csv, "bootstrap_poverty_2023_results_regular_split.csv")
pd.DataFrame({"bootstrap_poverty_rate": bootstrap_poverty_rates}).to_csv(bootstrap_results_path, index=False)

print(f"\n✅ Les résultats bootstrap ont été sauvegardés dans : {bootstrap_results_path}")

# Mesurer le temps d'exécution total
end_time = time.time()
elapsed_time = end_time - start_time

# Affichage du temps d'exécution
print("\n🔹 Temps total d'exécution :")
print(f"⏳ {elapsed_time:.2f} secondes")
print(f"⏳ {elapsed_time / 60:.2f} minutes")